# Phase 3 - Retrieval and Indexing

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [15]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ROOT,
    RAW,
    CFPB_RAW,
    ECFR_RAW,
    PDF_RAW,
    PROC,
    UA,
    INDEX
)

print("Project root :", ROOT)
print("Raw data     :", RAW)
print("CFPB         :", CFPB_RAW)
print("eCFR         :", ECFR_RAW)
print("PDF          :", PDF_RAW)
print("Processed    :", PROC)
print("Index    :", INDEX)

assert CFPB_RAW.exists()
assert ECFR_RAW.exists()
assert PDF_RAW.exists()

print("\nPASS: Phase 1 directories are ready.")

Project root : /home/sagemaker-user/complaintiq
Raw data     : /home/sagemaker-user/complaintiq/data/raw
CFPB         : /home/sagemaker-user/complaintiq/data/raw/cfpb
eCFR         : /home/sagemaker-user/complaintiq/data/raw/ecfr
PDF          : /home/sagemaker-user/complaintiq/data/raw/pdf
Processed    : /home/sagemaker-user/complaintiq/data/processed
Index    : /home/sagemaker-user/complaintiq/index

PASS: Phase 1 directories are ready.


In [2]:

from pathlib import Path
import os, sys, json, csv, hashlib, time, re
import numpy as np
import pandas as pd

PROJECT = Path("/home/sagemaker-user/complaintiq")
DATA = PROJECT / "data"
PROC = DATA / "processed"
INDEX = PROJECT / "index"
REPORTS = PROJECT / "reports"
EVAL = PROJECT / "eval"

for p in (PROJECT, DATA, PROC):
    if not p.exists():
        raise FileNotFoundError(f"Missing required directory: {p}")

for p in (INDEX, INDEX / "embed_cache", REPORTS, EVAL):
    p.mkdir(parents=True, exist_ok=True)

REGION = "us-east-1"
BUCKET = "sagemaker-complaintiq-456427397380"
S3_INDEX = f"s3://{BUCKET}/complaintiq/index/"

print("Project:", PROJECT)
print("Region:", REGION)
print("Index:", INDEX)
print("S3:", S3_INDEX)


Project: /home/sagemaker-user/complaintiq
Region: us-east-1
Index: /home/sagemaker-user/complaintiq/index
S3: s3://sagemaker-complaintiq-456427397380/complaintiq/index/


In [4]:
import pandas as pd
from pathlib import Path

PROJECT = Path("/home/sagemaker-user/complaintiq")
PROC = PROJECT / "data" / "processed"

INDEX_DIR = PROJECT / "index"

S3_INDEX = (
    "s3://sagemaker-complaintiq-456427397380/"
    "complaintiq/index/"
)

required = {
    "document_chunks": PROC / "document_chunks.jsonl",
    "complaint_pool": PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv",
    "june_holdout": PROC / "cfpb_holdout_june_2026.csv"
}

# Verify files
for name, path in required.items():
    print(
        "PASS" if path.exists() else "MISSING",
        "|", name, "|", path
    )
    assert path.exists(), f"Missing {path}"

# JSONL: physical lines represent individual JSON records
doc_count = sum(
    1 for _ in open(required["document_chunks"], encoding="utf-8")
)

# CSV: use Pandas because narratives may contain embedded newlines
pool_df = pd.read_csv(required["complaint_pool"])
holdout_df = pd.read_csv(required["june_holdout"])

pool_count = len(pool_df)
holdout_count = len(holdout_df)

print("\nDocument chunks:", doc_count)
print("Jan-May pool:", pool_count)
print("June holdout:", holdout_count)

# Validate expected counts
assert doc_count == 1784
assert pool_count == 31813
assert holdout_count == 6110

# Validate unique complaint IDs
assert pool_df["Complaint ID"].is_unique
assert holdout_df["Complaint ID"].is_unique

# Validate date separation
pool_dates = pd.to_datetime(pool_df["Date received"])
holdout_dates = pd.to_datetime(holdout_df["Date received"])

assert pool_dates.max() <= pd.Timestamp("2026-05-31")
assert holdout_dates.min() >= pd.Timestamp("2026-06-01")

# Validate no complaint ID overlap
assert set(pool_df["Complaint ID"]).isdisjoint(
    set(holdout_df["Complaint ID"])
)

print("\nPASS — Phase 2 inputs verified.")

PASS | document_chunks | /home/sagemaker-user/complaintiq/data/processed/document_chunks.jsonl
PASS | complaint_pool | /home/sagemaker-user/complaintiq/data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv
PASS | june_holdout | /home/sagemaker-user/complaintiq/data/processed/cfpb_holdout_june_2026.csv

Document chunks: 1784
Jan-May pool: 31813
June holdout: 6110

PASS — Phase 2 inputs verified.


In [4]:

%pip install -q boto3 faiss-cpu rank-bm25 tqdm


Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
import numpy as np

print("PASS — JSON and NumPy imported.")

PASS — JSON and NumPy imported.


In [11]:
import boto3

AWS_REGION = "us-east-1"

session = boto3.Session(region_name=AWS_REGION)

brt = session.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION
)

print("AWS Account:", session.client("sts").get_caller_identity()["Account"])
print("AWS Region:", AWS_REGION)
print("Bedrock Runtime client initialized successfully.")

AWS Account: 456427397380
AWS Region: us-east-1
Bedrock Runtime client initialized successfully.


In [12]:

test_body = json.dumps({
    "inputText": "A consumer reports an electronic funds transfer error.",
    "dimensions": 1024,
    "normalize": True
})

response = brt.invoke_model(
    modelId="amazon.titan-embed-text-v2:0",
    body=test_body,
    contentType="application/json",
    accept="application/json"
)

test_vector = np.asarray(
    json.loads(response["body"].read())["embedding"],
    dtype="float32"
)

print("Vector shape:", test_vector.shape)
print("Vector norm:", np.linalg.norm(test_vector))

assert test_vector.shape == (1024,)
assert np.isfinite(test_vector).all()

print("PASS — Titan Text Embeddings V2 is accessible.")


Vector shape: (1024,)
Vector norm: 1.0
PASS — Titan Text Embeddings V2 is accessible.


In [16]:

CACHE = INDEX / "embed_cache"
CACHE.mkdir(parents=True, exist_ok=True)

def cache_path(text, dim=1024):
    key = hashlib.sha1(
        f"{dim}|{text}".encode("utf-8")
    ).hexdigest()
    return CACHE / f"{key}.npy"


def embed_one(text, dim=1024):
    text = str(text or "").strip() or " "
    path = cache_path(text, dim)

    if path.exists():
        return np.load(path).astype("float32")

    body = json.dumps({
        "inputText": text[:30000],
        "dimensions": dim,
        "normalize": True
    })

    response = brt.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=body,
        contentType="application/json",
        accept="application/json"
    )

    vector = np.asarray(
        json.loads(response["body"].read())["embedding"],
        dtype="float32"
    )

    if vector.shape != (dim,):
        raise ValueError(f"Unexpected vector shape: {vector.shape}")

    if not np.isfinite(vector).all():
        raise ValueError("Embedding contains NaN or Inf")

    temp_path = path.with_suffix(".tmp.npy")
    np.save(temp_path, vector)
    os.replace(temp_path, path)

    return vector


def embed(texts, workers=3, dim=1024, desc="Embedding"):
    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc
            )
        )

    return np.vstack(vectors).astype("float32")


print("Cache directory:", CACHE)
print("Existing cached embeddings:", len(list(CACHE.glob("*.npy"))))


Cache directory: /home/sagemaker-user/complaintiq/index/embed_cache
Existing cached embeddings: 0


In [17]:

def read_jsonl(path):
    records = []

    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(
                        f"Invalid JSON at line {line_no}"
                    ) from exc

    return records


reg_chunks = read_jsonl(PROC / "document_chunks.jsonl")

print("Loaded document chunks:", len(reg_chunks))
assert len(reg_chunks) == 1784

for i, rec in enumerate(reg_chunks):
    if not rec.get("text"):
        raise ValueError(f"Chunk {i} has no text")

    if not rec.get("chunk_id"):
        doc = rec.get("doc_id") or rec.get("source") or "document"
        page = f"-p{rec['page']}" if rec.get("page") else ""
        rec["chunk_id"] = f"{doc}{page}-generated-{i}"

chunk_ids = [r["chunk_id"] for r in reg_chunks]

assert len(chunk_ids) == len(set(chunk_ids))

print("Unique chunk IDs:", len(set(chunk_ids)))
print("Example ID:", reg_chunks[0]["chunk_id"])
print("Available metadata:", sorted(reg_chunks[0].keys()))


Loaded document chunks: 1784
Unique chunk IDs: 1784
Example ID: 12CFR1002_1002.1_chunk_0001
Available metadata: ['char_count', 'chunk_id', 'chunk_number', 'citation', 'source_file', 'source_id', 'source_type', 'text', 'title']


In [18]:

chunk_lengths = np.array([len(r["text"]) for r in reg_chunks])

print(pd.Series(chunk_lengths).describe())

display(pd.DataFrame([
    {
        "chunk_id": r.get("chunk_id"),
        "source": r.get("source"),
        "doc_id": r.get("doc_id"),
        "page": r.get("page"),
        "section": r.get("section"),
        "characters": len(r["text"]),
        "preview": r["text"][:350]
    }
    for r in reg_chunks[:5]
]))


count    1784.000000
mean     2206.849215
std       740.341853
min        74.000000
25%      1834.500000
50%      2439.500000
75%      2817.000000
max      2999.000000
dtype: float64


,chunk_id,source,doc_id,page,section,characters,preview
0,12CFR1002_1002.1_chunk_0001,None,None,None,None,1824,"(a) Authority and scope. This part, known as R..."
1,12CFR1002_1002.2_chunk_0001,None,None,None,None,2758,"For the purposes of this part, unless the cont..."
2,12CFR1002_1002.2_chunk_0002,None,None,None,None,2697,"erm includes guarantors, sureties, endorsers, ..."
3,12CFR1002_1002.2_chunk_0003,None,None,None,None,2922,it may be made. A person is not a creditor reg...
4,12CFR1002_1002.2_chunk_0004,None,None,None,None,2692,he creditor is unable during the development p...


In [42]:
import os
import json
import hashlib
import boto3
import numpy as np
import pandas as pd
import re
import faiss
import numpy as np
from rank_bm25 import BM25Okapi


from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

print("PASS — All required imports loaded.")

PASS — All required imports loaded.


In [25]:
def embed(texts, workers=3, dim=1024, desc="Embedding"):

    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:

        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc,
                dynamic_ncols=True
            )
        )

    return np.vstack(vectors).astype("float32")


print("PASS — Embedding function updated.")

PASS — Embedding function updated.


In [26]:
reg_texts = [r["text"] for r in reg_chunks]

reg_vectors = embed(
    reg_texts,
    workers=3,
    desc="Regulatory/document embeddings"
)

assert reg_vectors.shape == (1784, 1024)
assert np.isfinite(reg_vectors).all()

reg_vectors_path = INDEX / "reg_vectors.npy"

np.save(reg_vectors_path, reg_vectors)

print("\nSaved:", reg_vectors_path)
print("Vector shape:", reg_vectors.shape)
print("Cache files:", len(list(CACHE.glob("*.npy"))))
print("Vector norm sample:", np.linalg.norm(reg_vectors[0]))

print("\nPASS — Regulatory document embeddings generated.")

Regulatory/document embeddings: 100%|██████████| 1784/1784 [01:25<00:00, 20.90it/s]


Saved: /home/sagemaker-user/complaintiq/index/reg_vectors.npy
Vector shape: (1784, 1024)
Cache files: 1782
Vector norm sample: 1.0

PASS — Regulatory document embeddings generated.


In [27]:

reg_metadata_path = INDEX / "reg_chunks.jsonl"

with open(reg_metadata_path, "w", encoding="utf-8") as f:
    for rec in reg_chunks:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved:", reg_metadata_path)


Saved: /home/sagemaker-user/complaintiq/index/reg_chunks.jsonl


In [28]:

cases_df = pd.read_csv(
    PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv",
    low_memory=False
)

cases_df.columns = [
    str(c).strip().lower().replace(" ", "_").replace("-", "_")
    for c in cases_df.columns
]

def find_col(candidates, required=True):
    for name in candidates:
        if name in cases_df.columns:
            return name

    if required:
        raise KeyError(
            f"Could not find {candidates}. "
            f"Available columns: {list(cases_df.columns)}"
        )

    return None


id_col = find_col(["complaint_id", "complaintid"])
product_col = find_col(["product"])
issue_col = find_col(["issue"])

narrative_col = find_col([
    "consumer_complaint_narrative",
    "complaint_narrative",
    "narrative"
])

response_col = find_col([
    "company_public_response",
    "company_response"
], required=False)

timely_col = find_col([
    "timely_response",
    "timely"
], required=False)

date_col = find_col([
    "date_received",
    "date"
], required=False)

print("Rows:", len(cases_df))
print("Resolved columns:")
print({
    "complaint_id": id_col,
    "product": product_col,
    "issue": issue_col,
    "narrative": narrative_col,
    "company_response": response_col,
    "timely": timely_col,
    "date": date_col
})

assert len(cases_df) == 31813
assert cases_df[id_col].astype(str).is_unique


Rows: 31813
Resolved columns:
{'complaint_id': 'complaint_id', 'product': 'product', 'issue': 'issue', 'narrative': 'consumer_complaint_narrative', 'company_response': 'company_public_response', 'timely': None, 'date': 'date_received'}


In [29]:

work = cases_df.copy()

work[narrative_col] = (
    work[narrative_col]
    .fillna("")
    .astype(str)
    .str.strip()
)

before = len(work)

work = work[
    work[narrative_col].str.len() > 0
].copy()

print("Original records:", before)
print("Records with narratives:", len(work))

case_records = []

for _, row in work.iterrows():
    case_records.append({
        "complaint_id": str(row[id_col]),
        "product": str(row[product_col]) if pd.notna(row[product_col]) else "",
        "issue": str(row[issue_col]) if pd.notna(row[issue_col]) else "",
        "company_response": (
            str(row[response_col])
            if response_col and pd.notna(row[response_col])
            else ""
        ),
        "timely": (
            str(row[timely_col])
            if timely_col and pd.notna(row[timely_col])
            else ""
        ),
        "date_received": (
            str(row[date_col])
            if date_col and pd.notna(row[date_col])
            else ""
        ),
        "text": row[narrative_col]
    })

assert len(case_records) > 3000

print("Prepared searchable complaints:", len(case_records))


Original records: 31813
Records with narratives: 31813
Prepared searchable complaints: 31813


In [30]:

case_frame = pd.DataFrame(case_records)

case_frame["_stratum"] = (
    case_frame["product"].astype(str)
    + " || "
    + case_frame["issue"].astype(str)
)

TARGET_CASES = 3000
SEED = 42

strata = case_frame.groupby("_stratum").size().sort_index()

raw_alloc = strata / strata.sum() * TARGET_CASES
alloc = np.floor(raw_alloc).astype(int)

remainder = TARGET_CASES - int(alloc.sum())

if remainder:
    order = (
        raw_alloc - alloc
    ).sort_values(ascending=False, kind="stable").index[:remainder]

    alloc.loc[order] += 1

alloc = alloc.clip(upper=strata)

shortfall = TARGET_CASES - int(alloc.sum())

while shortfall > 0:
    eligible = strata[alloc < strata]

    if eligible.empty:
        raise ValueError("Cannot allocate the requested sample size.")

    for key in eligible.sort_values(ascending=False).index:
        if shortfall == 0:
            break

        alloc.loc[key] += 1
        shortfall -= 1

sampled_parts = []

for key, group in case_frame.groupby("_stratum", sort=True):
    n = int(alloc.loc[key])

    if n:
        sampled_parts.append(
            group.sample(n=n, random_state=SEED)
        )

case_sample_df = pd.concat(
    sampled_parts,
    ignore_index=True
)

case_sample_df = (
    case_sample_df
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

case_sample_df = case_sample_df.drop(columns=["_stratum"])

assert len(case_sample_df) == 3000
assert case_sample_df["complaint_id"].is_unique

case_chunks = case_sample_df.to_dict(orient="records")

print("Number of strata:", len(strata))
print("Sample size:", len(case_chunks))
print("Unique complaint IDs:", case_sample_df["complaint_id"].nunique())

display(
    case_sample_df.groupby(["product", "issue"])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .rename("sample_count")
    .reset_index()
)


Number of strata: 26
Sample size: 3000
Unique complaint IDs: 3000


,product,issue,sample_count
0,Checking or savings account,Managing an account,857
1,Credit card,Problem with a purchase shown on your statement,519
2,Checking or savings account,Problem with a lender or other company chargin...,238
3,Checking or savings account,Closing an account,202
4,Credit card,"Other features, terms, or problems",187
5,Credit card,Fees or interest,177
6,Credit card,Getting a credit card,146
7,Checking or savings account,Opening an account,107
8,Credit card,Closing your account,106
9,Checking or savings account,Problem caused by your funds being low,97


In [31]:

holdout_df = pd.read_csv(
    PROC / "cfpb_holdout_june_2026.csv",
    low_memory=False
)

holdout_df.columns = [
    str(c).strip().lower().replace(" ", "_").replace("-", "_")
    for c in holdout_df.columns
]

holdout_id_col = (
    "complaint_id"
    if "complaint_id" in holdout_df.columns
    else "complaintid"
)

sample_ids = set(case_sample_df["complaint_id"].astype(str))
holdout_ids = set(holdout_df[holdout_id_col].astype(str))

overlap = sample_ids.intersection(holdout_ids)

print("Sample IDs:", len(sample_ids))
print("June holdout IDs:", len(holdout_ids))
print("Overlap:", len(overlap))

assert len(overlap) == 0

print("PASS — no June holdout leakage.")


Sample IDs: 3000
June holdout IDs: 6110
Overlap: 0
PASS — no June holdout leakage.


In [32]:

def case_search_text(rec):
    return "\n".join([
        f"Product: {rec.get('product', '')}",
        f"Issue: {rec.get('issue', '')}",
        f"Consumer complaint: {rec.get('text', '')}"
    ])

case_texts = [
    case_search_text(r)
    for r in case_chunks
]

assert len(case_texts) == 3000

print(case_texts[0][:700])


Product: Credit card
Issue: Fees or interest
Consumer complaint: This is a follow-up to complaint # XXXX against SYNCB/CareCredit. In response to my original complaint, Synchrony opened a merchant dispute ( Claim # XXXX ) contacting the dental office. This is incorrect the dental services are not in dispute. My complaint is solely about the {$2700.00} deferred interest charge from XXXX XXXX. On XX/XX/XXXX, I financed {$4300.00} through CareCredit under a 24-month, and had a 0 % interest plan ( I had zero interest for 2 years ). The payment plan was set up by the XXXX office- I never chose my monthly payment amount and was never told what amount was required to avoid deferred interest. I made


In [33]:

case_vectors = embed(
    case_texts,
    workers=3,
    desc="Complaint embeddings"
)

assert case_vectors.shape == (3000, 1024)
assert np.isfinite(case_vectors).all()

case_vectors_path = INDEX / "case_vectors.npy"
np.save(case_vectors_path, case_vectors)

print("Saved:", case_vectors_path)
print("Shape:", case_vectors.shape)


Complaint embeddings: 100%|██████████| 3000/3000 [01:51<00:00, 26.91it/s]


Saved: /home/sagemaker-user/complaintiq/index/case_vectors.npy
Shape: (3000, 1024)


In [34]:

case_metadata_path = INDEX / "case_chunks.jsonl"

with open(case_metadata_path, "w", encoding="utf-8") as f:
    for rec in case_chunks:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved:", case_metadata_path)


Saved: /home/sagemaker-user/complaintiq/index/case_chunks.jsonl


In [35]:

def tok(s):
    return re.findall(
        r"[a-z0-9]+(?:\.[0-9]+)?",
        str(s).lower()
    )


class HybridIndex:

    def __init__(self, chunks, vectors):
        self.chunks = list(chunks)
        self.vectors = np.asarray(vectors, dtype="float32")

        if len(self.vectors) != len(self.chunks):
            raise ValueError("Vector/metadata count mismatch")

        if not np.isfinite(self.vectors).all():
            raise ValueError("Invalid vectors")

        self.faiss = faiss.IndexFlatIP(
            self.vectors.shape[1]
        )

        self.faiss.add(self.vectors)

        self.bm25 = BM25Okapi([
            tok(c.get("text", ""))
            for c in self.chunks
        ])

    def search(self, query, k=8, pool=40, rrf_k=60):

        if not query or not str(query).strip():
            raise ValueError("Query cannot be blank")

        pool = min(pool, len(self.chunks))

        qv = embed([query], workers=1)

        _, dense_ids = self.faiss.search(qv, pool)

        sparse_scores = self.bm25.get_scores(tok(query))

        sparse_ids = np.argsort(
            sparse_scores
        )[::-1][:pool]

        scores = defaultdict(float)

        for ranking in (dense_ids[0], sparse_ids):
            for rank, idx in enumerate(ranking):
                if int(idx) >= 0:
                    scores[int(idx)] += (
                        1.0 / (rrf_k + rank + 1)
                    )

        # Prioritize exact CFR citation matches.
        citation = re.search(
            r"\b(\d{1,2})\s*CFR\s*(\d{3,4}\.\d+)\b",
            query,
            re.I
        )

        if citation:
            target = (
                f"{citation.group(1)} CFR "
                f"{citation.group(2)}"
            ).lower()

            compact = (
                f"{citation.group(1)}cfr"
                f"{citation.group(2)}"
            ).lower()

            for idx, rec in enumerate(self.chunks):

                haystack = " ".join([
                    str(rec.get("chunk_id", "")),
                    str(rec.get("section", "")),
                    str(rec.get("text", ""))
                ]).lower()

                normalized = re.sub(
                    r"\s+", " ", haystack
                )

                compact_haystack = re.sub(
                    r"[^a-z0-9.]", "", haystack
                )

                if (
                    target in normalized
                    or compact in compact_haystack
                ):
                    scores[idx] += 1.0

        top = sorted(
            scores,
            key=scores.get,
            reverse=True
        )[:k]

        return [
            {
                **self.chunks[i],
                "rrf": round(float(scores[i]), 6)
            }
            for i in top
        ]


print("HybridIndex defined.")


HybridIndex defined.


In [37]:
import faiss

print("FAISS version:", faiss.__version__)
print("PASS — FAISS imported successfully.")

FAISS version: 1.15.1
PASS — FAISS imported successfully.


In [43]:
REG_INDEX = HybridIndex(
    reg_chunks,
    reg_vectors
)

CASE_INDEX = HybridIndex(
    case_chunks,
    case_vectors
)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("PASS — Regulatory index:", REG_INDEX.faiss.ntotal)
print("PASS — Complaint index:", CASE_INDEX.faiss.ntotal)

PASS — Regulatory index: 1784
PASS — Complaint index: 3000


In [47]:
from collections import defaultdict

print("PASS — defaultdict imported successfully.")

PASS — defaultdict imported successfully.


In [48]:

reg_queries = [
    {
        "query": "error resolution timeline electronic fund transfer",
        "expected": ["1005.11", "12cfr1005.11"],
        "purpose": "Regulation E error resolution"
    },
    {
        "query": "billing error notice credit card",
        "expected": ["1026.13", "12cfr1026.13"],
        "purpose": "Credit card billing error"
    },
    {
        "query": "fair credit reporting accuracy investigation dispute",
        "expected": ["1022", "fcra"],
        "purpose": "Credit reporting dispute"
    },
    {
        "query": "debt collector validation notice dispute time period",
        "expected": ["1006", "validation"],
        "purpose": "Debt collection validation"
    },
    {
        "query": "equal credit opportunity adverse action notice",
        "expected": ["1002", "adverse action"],
        "purpose": "ECOA adverse action"
    }
]

reg_test_rows = []

for item in reg_queries:

    results = REG_INDEX.search(
        item["query"],
        k=5,
        pool=40,
        rrf_k=60
    )

    combined = " ".join(
        " ".join([
            str(r.get("chunk_id", "")),
            str(r.get("section", "")),
            str(r.get("text", ""))
        ])
        for r in results
    ).lower()

    hit = any(
        term.lower() in combined
        for term in item["expected"]
    )

    reg_test_rows.append({
        "query": item["query"],
        "purpose": item["purpose"],
        "expected_terms": "|".join(item["expected"]),
        "expected_found_top5": hit,
        "top5_chunk_ids": "|".join(
            str(r.get("chunk_id"))
            for r in results
        )
    })

    print("\nQUERY:", item["query"])
    print("Expected found in top 5:", hit)

    for rank, result in enumerate(results, 1):
        print(
            rank,
            result.get("chunk_id"),
            "RRF:", result.get("rrf"),
            result.get("text", "")[:180].replace("\n", " ")
        )

reg_smoke_df = pd.DataFrame(reg_test_rows)

display(reg_smoke_df)

reg_smoke_path = INDEX / "reg_retrieval_smoke_tests.csv"
reg_smoke_df.to_csv(reg_smoke_path, index=False)

print("Saved:", reg_smoke_path)


Embedding: 100%|██████████| 1/1 [00:00<00:00, 6543.38it/s]



QUERY: error resolution timeline electronic fund transfer
Expected found in top 5: True
1 12CFR1005_1005.11_chunk_0001 RRF: 0.03101 (a) Definition of error —(1) Types of transfers or inquiries covered. The term “error” means: (i) An unauthorized electronic fund transfer; (ii) An incorrect electronic fund transf
2 12CFR1005_1005.11_chunk_0003 RRF: 0.030886 nt was made. (ii) The applicable time is 90 days in place of 45 days under paragraph (c)(2) of this section, for completing an investigation, if a notice of error involves an elect
3 12CFR1005_1005.33_chunk_0004 RRF: 0.030579 ii)(A)( 1 ) and (c)(2)(ii)(B) of this section within three business days of providing the report required by paragraph (c)(1) or (d)(1) of this section except that the provider may
4 12CFR1005_1005.14_chunk_0002 RRF: 0.030303 sional credit in accordance with § 1005.11(c)(2)(ii). (iii) If the service provider determines an error occurred, it shall transfer funds to or from the consumer's account, in the 
5 12CFR1

Embedding: 100%|██████████| 1/1 [00:00<00:00,  3.67it/s]



QUERY: billing error notice credit card
Expected found in top 5: True
1 12CFR1026_1026.13_chunk_0001 RRF: 0.032787 (a) Definition of billing error. For purposes of this section, the term billing error means: (1) A reflection on or with a periodic statement of an extension of credit that is not 
2 12CFR1026_1026.13_chunk_0002 RRF: 0.032258 ion. Until a billing error is resolved under paragraph (e) or (f) of this section, the following rules apply: (1) Consumer's right to withhold disputed amount; collection action pr
3 12CFR1026_1026.13_chunk_0003 RRF: 0.031498 or determines that a billing error occurred as asserted, it shall within the time limits in paragraph (c)(2) of this section: (1) Correct the billing error and credit the consumer'
4 12CFR1026_1026.13_chunk_0004 RRF: 0.030159 ) Promptly reports that the amount or account is in dispute; (ii) Mails or delivers to the consumer (at the same time the report is made) a written notice of the name and address o
5 12CFR1026_1026.7_chunk_

Embedding: 100%|██████████| 1/1 [00:00<00:00,  8.35it/s]



QUERY: fair credit reporting accuracy investigation dispute
Expected found in top 5: True
1 exam_manual_v2_2012_page_0664_chunk_0001 RRF: 0.03101 CFPB Examination Procedures FCRA c. provided the consumer reporting agency with corrections or additional information to make information complete and accurate, and thereafter did 
2 sh_issue22_2020-09_page_0004_chunk_0001 RRF: 0.029727 3 SUPERVISORY HIGHLIGHTS, ISSUE 22 - SUMMER 2020 2. Supervisory observations Recent supervisory observations are reported in the areas of consumer reporting, debt collection, depos
3 exam_manual_v2_2012_page_0581_chunk_0001 RRF: 0.028543 CFPB Consumer Laws and Regulations FCRA Fair Credit Reporting Act1 The Fair Credit Reporting Act (FCRA)2 became effective on April 25, 1971. The FCRA is a part of a group of acts c
4 exam_manual_v2_2012_page_0636_chunk_0001 RRF: 0.028405 CFPB Consumer Laws and Regulations FCRA The person must complete the required investigations, reviews, and reports within 30 days. The perso

... (2 outputs left out)

Embedding: 100%|██████████| 1/1 [00:00<00:00,  8.73it/s]


QUERY: equal credit opportunity adverse action notice
Expected found in top 5: True
1 12CFR1002_1002.9_chunk_0002 RRF: 0.032522 the disclosure contains the information required by paragraph (a)(2)(ii) of this section and the ECOA notice specified in paragraph (b)(1) of this section; (C) For an application m
2 exam_manual_v2_2012_page_0219_chunk_0001 RRF: 0.032266 CFPB Examination Procedures ECOA CFPB Manual V.2 (October 2012) Procedures 17 Yes No Basis of Conclusion b. Within 30 days after taking adverse action on an incomplete application,
3 exam_manual_v2_2012_page_0199_chunk_0001 RRF: 0.031281 CFPB Consumer Laws and Regulations ECOA Under at least two circumstances, the creditor need not comply with the 30-day notification rule: • The creditor must notify an applicant of
4 12CFR1002_1002.9_chunk_0001 RRF: 0.029644 (a) Notification of action taken, ECOA notice, and statement of specific reasons —(1) When notification is required. A creditor shall notify an applicant of action taken 

,query,purpose,expected_terms,expected_found_top5,top5_chunk_ids
0,error resolution timeline electronic fund tran...,Regulation E error resolution,1005.11|12cfr1005.11,True,12CFR1005_1005.11_chunk_0001|12CFR1005_1005.11...
1,billing error notice credit card,Credit card billing error,1026.13|12cfr1026.13,True,12CFR1026_1026.13_chunk_0001|12CFR1026_1026.13...
2,fair credit reporting accuracy investigation d...,Credit reporting dispute,1022|fcra,True,exam_manual_v2_2012_page_0664_chunk_0001|sh_is...
3,debt collector validation notice dispute time ...,Debt collection validation,1006|validation,True,12CFR1006_1006.34_chunk_0003|12CFR1006_1006.34...
4,equal credit opportunity adverse action notice,ECOA adverse action,1002|adverse action,True,12CFR1002_1002.9_chunk_0002|exam_manual_v2_201...


Saved: /home/sagemaker-user/complaintiq/index/reg_retrieval_smoke_tests.csv


In [49]:

exact_query = "12 CFR 1005.11 error resolution procedures"

exact_results = REG_INDEX.search(
    exact_query,
    k=5,
    pool=40,
    rrf_k=60
)

display(pd.DataFrame([
    {
        "rank": i,
        "chunk_id": r.get("chunk_id"),
        "section": r.get("section"),
        "source": r.get("source"),
        "rrf": r.get("rrf"),
        "preview": r.get("text", "")[:350]
    }
    for i, r in enumerate(exact_results, 1)
]))

first_id = str(
    exact_results[0].get("chunk_id", "")
).lower()

first_text = str(
    exact_results[0].get("text", "")
).lower()

exact_pass = (
    "1005.11" in first_id
    or "1005.11" in first_text
)

print("Exact CFR section ranked first:", exact_pass)

if not exact_pass:
    print("REVIEW REQUIRED — inspect the result metadata and citation matching.")


Embedding: 100%|██████████| 1/1 [00:00<00:00,  5.58it/s]


,rank,chunk_id,section,source,rrf,preview
0,1,exam_manual_v2_2012_page_0700_chunk_0001,None,None,1.032266,CFPB Consumer Laws and Regulations EFTA An err...
1,2,exam_manual_v2_2012_page_0699_chunk_0002,None,None,1.030550,"an EFT, for example, when the card is used to ..."
2,3,exam_manual_v2_2012_page_0734_chunk_0001,None,None,1.030536,CFPB Examination Checklist EFTA CFPB Manual V....
3,4,12CFR1026_1026.13_chunk_0004,None,None,1.027402,) Promptly reports that the amount or account ...
4,5,exam_manual_v2_2012_page_0735_chunk_0001,None,None,1.027149,Yes No NA CFPB Examination Checklist EFTA 39. ...


Exact CFR section ranked first: True


In [50]:

case_query = (
    "My debit card transfer was unauthorized and "
    "the bank did not resolve the error within the required time."
)

case_results = CASE_INDEX.search(
    case_query,
    k=5,
    pool=40,
    rrf_k=60
)

case_smoke = []

for rank, rec in enumerate(case_results, 1):
    case_smoke.append({
        "rank": rank,
        "complaint_id": rec.get("complaint_id"),
        "product": rec.get("product"),
        "issue": rec.get("issue"),
        "company_response": rec.get("company_response"),
        "timely": rec.get("timely"),
        "rrf": rec.get("rrf"),
        "narrative_preview": rec.get("text", "")[:500]
    })

display(pd.DataFrame(case_smoke))

case_smoke_path = INDEX / "case_retrieval_smoke_tests.jsonl"

with open(case_smoke_path, "w", encoding="utf-8") as f:
    for row in case_smoke:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", case_smoke_path)


Embedding: 100%|██████████| 1/1 [00:00<00:00,  7.94it/s]


,rank,complaint_id,product,issue,company_response,timely,rrf,narrative_preview
0,1,20778845,Checking or savings account,Managing an account,,,0.032266,"On XXXX of XXXX, I discovered an unauthorized ..."
1,2,18834896,Checking or savings account,Managing an account,,,0.030366,I am writing to formally demand immediate reso...
2,3,21896172,Checking or savings account,Problem with a lender or other company chargin...,,,0.029199,CFPB Complaint ( or Bank Escalation Message ) ...
3,4,19040619,Checking or savings account,Managing an account,Company has responded to the consumer and the ...,,0.028043,I am filing a complaint regarding USAAs mishan...
4,5,21283530,Checking or savings account,Managing an account,Company has responded to the consumer and the ...,,0.027973,"b""Back on XXXX XXXX I found out that on my bus..."


Saved: /home/sagemaker-user/complaintiq/index/case_retrieval_smoke_tests.jsonl


In [53]:
import datetime
import time
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


artifact_paths = [
    reg_vectors_path,
    reg_metadata_path,
    case_vectors_path,
    case_metadata_path,
    reg_smoke_path,
    case_smoke_path
]

manifest = {
    "phase": "3",
    "project": "ComplaintIQ",
    "created_utc": time.strftime(
        "%Y-%m-%dT%H:%M:%SZ",
        time.gmtime()
    ),
    "region": REGION,
    "embedding_model": "amazon.titan-embed-text-v2:0",
    "embedding_dimensions": 1024,
    "dense_index": "FAISS IndexFlatIP",
    "sparse_index": "BM25Okapi",
    "fusion": "Reciprocal Rank Fusion",
    "rrf_k": 60,
    "complaint_sample_size": len(case_chunks),
    "complaint_sample_seed": SEED,
    "source_counts": {
        "document_chunks": len(reg_chunks),
        "jan_may_pool": len(cases_df),
        "june_holdout": len(holdout_df)
    },
    "june_holdout_indexed": False,
    "exact_citation_test_passed": bool(exact_pass),
    "regulatory_smoke_tests": reg_test_rows,
    "artifacts": {
        p.name: {
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p)
        }
        for p in artifact_paths
    }
}

manifest_path = INDEX / "phase3_retrieval_manifest.json"

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("Saved:", manifest_path)
print(json.dumps(manifest, indent=2)[:4000])


Saved: /home/sagemaker-user/complaintiq/index/phase3_retrieval_manifest.json
{
  "phase": "3",
  "project": "ComplaintIQ",
  "created_utc": "2026-11-06T06:01:18Z",
  "region": "us-east-1",
  "embedding_model": "amazon.titan-embed-text-v2:0",
  "embedding_dimensions": 1024,
  "dense_index": "FAISS IndexFlatIP",
  "sparse_index": "BM25Okapi",
  "fusion": "Reciprocal Rank Fusion",
  "rrf_k": 60,
  "complaint_sample_size": 3000,
  "complaint_sample_seed": 42,
  "source_counts": {
    "document_chunks": 1784,
    "jan_may_pool": 31813,
    "june_holdout": 6110
  },
  "june_holdout_indexed": false,
  "exact_citation_test_passed": true,
  "regulatory_smoke_tests": [
    {
      "query": "error resolution timeline electronic fund transfer",
      "purpose": "Regulation E error resolution",
      "expected_terms": "1005.11|12cfr1005.11",
      "expected_found_top5": true,
      "top5_chunk_ids": "12CFR1005_1005.11_chunk_0001|12CFR1005_1005.11_chunk_0003|12CFR1005_1005.33_chunk_0004|12CFR1005_10

In [54]:

assert reg_vectors.shape == (1784, 1024)
assert len(reg_chunks) == 1784

assert case_vectors.shape == (3000, 1024)
assert len(case_chunks) == 3000

assert len(overlap) == 0

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

assert manifest_path.exists()
assert reg_smoke_path.exists()
assert case_smoke_path.exists()

print("PASS | Regulatory vectors and metadata align")
print("PASS | Complaint vectors and metadata align")
print("PASS | 3,000 sampled complaints")
print("PASS | No June holdout leakage")
print("PASS | Both FAISS indexes built")
print("PASS | Smoke-test artifacts saved")
print("PASS | Phase 3 manifest saved")

if not all(reg_smoke_df["expected_found_top5"]):
    print("REVIEW | Some expected regulatory terms were not found in top five.")

if not exact_pass:
    print("REVIEW | Exact CFR citation test did not pass.")


PASS | Regulatory vectors and metadata align
PASS | Complaint vectors and metadata align
PASS | 3,000 sampled complaints
PASS | No June holdout leakage
PASS | Both FAISS indexes built
PASS | Smoke-test artifacts saved
PASS | Phase 3 manifest saved


In [55]:

import subprocess

subprocess.run(
    [
        "aws", "s3", "sync",
        str(INDEX),
        S3_INDEX,
        "--region", REGION
    ],
    check=True
)

print("PASS — Index directory synced to:", S3_INDEX)


upload: ../index/case_retrieval_smoke_tests.jsonl to s3://sagemaker-complaintiq-456427397380/complaintiq/index/case_retrieval_smoke_tests.jsonl
upload: ../index/case_chunks.jsonl to s3://sagemaker-complaintiq-456427397380/complaintiq/index/case_chunks.jsonl
upload: ../index/embed_cache/0011056913cf0ec69391c1c183afc1789e37f17c.npy to s3://sagemaker-complaintiq-456427397380/complaintiq/index/embed_cache/0011056913cf0ec69391c1c183afc1789e37f17c.npy
upload: ../index/embed_cache/0017c304719e1cc8f0cc481667d763aea846222c.npy to s3://sagemaker-complaintiq-456427397380/complaintiq/index/embed_cache/0017c304719e1cc8f0cc481667d763aea846222c.npy
upload: ../index/embed_cache/001e4e4a6fe6b8163f55a1c759707431343e69c5.npy to s3://sagemaker-complaintiq-456427397380/complaintiq/index/embed_cache/001e4e4a6fe6b8163f55a1c759707431343e69c5.npy
upload: ../index/embed_cache/003cc06b5d7b621547ad0535a0297fa1376a8fc1.npy to s3://sagemaker-complaintiq-456427397380/complaintiq/index/embed_cache/003cc06b5d7b621547a

In [57]:
import boto3

REGION = "us-east-1"
BUCKET = "sagemaker-complaintiq-456427397380"

s3 = boto3.client("s3", region_name=REGION)

print("AWS Region:", REGION)
print("S3 Bucket:", BUCKET)

# Verify bucket access
s3.head_bucket(Bucket=BUCKET)

print("PASS — S3 bucket is accessible.")

AWS Region: us-east-1
S3 Bucket: sagemaker-complaintiq-456427397380
PASS — S3 bucket is accessible.


In [58]:

s3 = boto3.client("s3", region_name=REGION)

required_s3_files = [
    "reg_vectors.npy",
    "reg_chunks.jsonl",
    "case_vectors.npy",
    "case_chunks.jsonl",
    "reg_retrieval_smoke_tests.csv",
    "case_retrieval_smoke_tests.jsonl",
    "phase3_retrieval_manifest.json"
]

for filename in required_s3_files:

    key = f"complaintiq/index/{filename}"

    head = s3.head_object(
        Bucket=BUCKET,
        Key=key
    )

    print(
        "PASS |",
        filename,
        "|",
        f"{head['ContentLength']:,} bytes"
    )

print("\nPASS — Required Phase 3 artifacts verified in S3.")


PASS | reg_vectors.npy | 7,307,392 bytes
PASS | reg_chunks.jsonl | 4,504,700 bytes
PASS | case_vectors.npy | 12,288,128 bytes
PASS | case_chunks.jsonl | 4,897,915 bytes
PASS | reg_retrieval_smoke_tests.csv | 1,344 bytes
PASS | case_retrieval_smoke_tests.jsonl | 3,705 bytes
PASS | phase3_retrieval_manifest.json | 3,430 bytes

PASS — Required Phase 3 artifacts verified in S3.


In [59]:

notes = f"""# ComplaintIQ — Phase 3 Retrieval Layer

## Objective
Retrieve relevant banking regulations/reference passages
and similar historical consumer complaints.

## Implementation
- Embeddings: Amazon Bedrock Titan Text Embeddings V2
- Dimensions: 1,024
- Dense retrieval: FAISS IndexFlatIP
- Keyword retrieval: BM25Okapi
- Fusion: Reciprocal Rank Fusion, k=60
- Cache: SHA1-keyed NumPy files
- Embedding concurrency: 3 workers

## Indexed corpus
- Regulatory/document chunks: {len(reg_chunks)}
- Deduplicated Jan-May complaint pool: {len(cases_df)}
- Stratified complaint sample: {len(case_chunks)}
- June holdout: {len(holdout_df)} (excluded)
- Sample seed: {SEED}

## Validation
- Five regulatory smoke tests saved.
- Exact 12 CFR 1005.11 test: {exact_pass}
- Similar complaint results saved.
- June holdout overlap: {len(overlap)}

## Why hybrid retrieval?
Dense retrieval identifies semantically similar passages.
BM25 matches exact terminology, decimal CFR section numbers,
and other lexical details. RRF combines both ranked lists.

## Limitations
Smoke tests are functional checks, not a comprehensive benchmark.
Phase 7 must evaluate recall@5 and compare dense-only,
BM25-only, and hybrid retrieval against a labeled golden set.
"""

notes_path = INDEX / "phase3_retrieval_notes.md"

notes_path.write_text(
    notes,
    encoding="utf-8"
)

print("Saved:", notes_path)


Saved: /home/sagemaker-user/complaintiq/index/phase3_retrieval_notes.md
